# 1. Nível Iniciante: Ticket Médio por Transação

- **O que é:** Quanto, em média, cada cliente gasta por compra na loja.
- **Lógica:** $\text{Ticket Médio} = \frac{\text{Faturamento Total}}{\text{Total de Transações}}$
- **O que treinar:** `.sum()`, `.count()`, `len()`, `.mean()` e formatação monetária.

In [1]:
import pandas as pd

# Carregando a base detalhada
df = pd.read_excel('../data/raw/loja_informatica_2026.xlsx', sheet_name='Dados Detalhados')
df['Data'] = pd.to_datetime(df['Data'], format='%d/%m/%Y')

# Calculando o faturamento por linha
df['Faturamento Total (R$)'] = df['Preço Unitário (R$)'] * df['Qtd Vendida']

df.head()

,ID Transação,Data,Mês,Produto,Categoria,Preço Unitário (R$),Qtd Vendida,Faturamento Total (R$),Estoque Atual,Status Estoque
0,TR1001,2026-02-27,Fevereiro,Notebook Gamer i7 16GB,Hardware,4608.70,4,18434.80,33,NaN
1,TR1002,2026-03-13,Março,Fonte 600W 80 Plus,Componentes,384.10,12,4609.20,74,NaN
2,TR1003,2026-02-14,Fevereiro,Headset Gamer 7.1,Periféricos,277.81,1,277.81,16,NaN
3,TR1004,2026-04-22,Abril,Processador Ryzen 7 5700X,Componentes,1400.75,1,1400.75,76,NaN
4,TR1005,2026-04-12,Abril,Fonte 600W 80 Plus,Componentes,406.00,9,3654.00,58,NaN


### Método 1: `.sum()` dividido por `.count()` ou `len()`

In [2]:
faturamento_total = df['Faturamento Total (R$)'].sum()
total_transacoes = df['ID Transação'].count()  # ou len(df)

ticket_medio_m1 = faturamento_total / total_transacoes

print(f"Faturamento Total: R$ {faturamento_total:,.2f}")
print(f"Total de Transações: {total_transacoes}")
print(f"Ticket Médio (Método 1): R$ {ticket_medio_m1:,.2f}")

Faturamento Total: R$ 4,620,144.91
Total de Transações: 500
Ticket Médio (Método 1): R$ 9,240.29


### Método 2: Média direta com `.mean()`

In [3]:
# Como cada linha representa uma transação, podemos tirar a média direta da coluna
ticket_medio_m2 = df['Faturamento Total (R$)'].mean()

print(f"Ticket Médio (Método 2 - .mean()): R$ {ticket_medio_m2:,.2f}")

Ticket Médio (Método 2 - .mean()): R$ 9,240.29


### Método 3 (Boas Práticas): Validação de Transações Únicas

> **Dica Analítica:** Em bases reais onde um mesmo pedido/transação pode ter múltiplos itens (várias linhas com o mesmo `ID Transação`), agrupar antes garante o cálculo correto do valor por carrinho/compra.

In [4]:
transacoes_unicas = df['ID Transação'].nunique()
total_linhas = len(df)

print(f"Total de Linhas: {total_linhas}")
print(f"Total de Transações Únicas: {transacoes_unicas}")

# Se cada ID for único, o resultado do groupby é idêntico:
ticket_medio_por_pedido = df.groupby('ID Transação')['Faturamento Total (R$)'].sum().mean()
print(f"Ticket Médio por Pedido Agrupado: R$ {ticket_medio_por_pedido:,.2f}")

Total de Linhas: 500
Total de Transações Únicas: 500
Ticket Médio por Pedido Agrupado: R$ 9,240.29


### 4. Simulação: E se um mesmo ID tivesse mais de uma linha (múltiplos itens por pedido)?

Quando um mesmo `ID Transação` aparece mais de uma vez:
1. **Cálculo ingênuo (`.mean()` ou `.sum() / len()`):** calcula a *média por item*, distorcendo o ticket da compra para baixo.
2. **Cálculo correto (com `.nunique()` ou `.groupby()`):** preserva a totalidade da compra somando os itens do mesmo pedido antes de tirar a média.

In [5]:
# Criando um exemplo simulado com múltiplos itens no mesmo ID de transação
df_exemplo = pd.DataFrame({
    'ID Transação': ['TR01', 'TR01', 'TR02', 'TR03', 'TR03', 'TR03'],
    'Produto': ['Notebook', 'Mouse', 'Monitor', 'Teclado', 'Headset', 'Webcam'],
    'Faturamento': [5000.0, 100.0, 1500.0, 200.0, 300.0, 400.0]
})

print("=== Base Simulada com Itens Múltiplos por Pedido ===")
display(df_exemplo)

# 1. Cálculo Ingênuo (ERRADO para ticket por compra):
ticket_errado = df_exemplo['Faturamento'].mean()

# 2. Cálculo Correto usando .nunique():
ticket_correto_nunique = df_exemplo['Faturamento'].sum() / df_exemplo['ID Transação'].nunique()

# 3. Cálculo Correto agrupando com .groupby():
totais_por_pedido = df_exemplo.groupby('ID Transação')['Faturamento'].sum()
ticket_correto_groupby = totais_por_pedido.mean()

print(f"\nTotal de Linhas (itens): {len(df_exemplo)}")
print(f"Total de Pedidos Reais (nunique): {df_exemplo['ID Transação'].nunique()}")
print(f"❌ Ticket Médio por Item (Ingênuo): R$ {ticket_errado:,.2f}")
print(f"✅ Ticket Médio por Pedido (.nunique): R$ {ticket_correto_nunique:,.2f}")
print(f"✅ Ticket Médio por Pedido (.groupby): R$ {ticket_correto_groupby:,.2f}")
print("\n--- Detalhe por Pedido (.groupby) ---")
print(totais_por_pedido)

=== Base Simulada com Itens Múltiplos por Pedido ===


,ID Transação,Produto,Faturamento
0,TR01,Notebook,5000.0
1,TR01,Mouse,100.0
2,TR02,Monitor,1500.0
3,TR03,Teclado,200.0
4,TR03,Headset,300.0
5,TR03,Webcam,400.0



Total de Linhas (itens): 6
Total de Pedidos Reais (nunique): 3
❌ Ticket Médio por Item (Ingênuo): R$ 1,250.00
✅ Ticket Médio por Pedido (.nunique): R$ 2,500.00
✅ Ticket Médio por Pedido (.groupby): R$ 2,500.00

--- Detalhe por Pedido (.groupby) ---
ID Transação
TR01    5100.0
TR02    1500.0
TR03     900.0
Name: Faturamento, dtype: float64
